# WPMW — Kaggle Runner

Thin launcher notebook for scripts in the
[wpmw](https://github.com/billpage/wpmw) repository (Wigner phase-space
crystal-lattice / extended Fokker–Planck research).

**Default target:** `demo_cat_state_microdynamics.py` — the free-particle cat
state via crystal-lattice microdynamics demo.

**Usage:**
1. Run **Cell 1 (Setup)** once per session — clones the repo and sets `WPMW_OUTPUT`.
2. Edit the `SCRIPT` and `ARGS` variables in **Cell 2 (Run)** if you want a
   different script, then execute.
3. PNG outputs land in `/kaggle/working/` and are displayed by **Cell 3**.

After pushing changes to GitHub, re-run Cell 1 to `git pull` the latest code.

**Notes:**
* Current wpmw scripts (including the default cat-state microdynamics demo)
  are pure NumPy + matplotlib and do not require a GPU. The GPU accelerator
  and CuPy install are kept in this runner because future multi-DOF
  simulations are expected to use them; both are no-ops for the current
  scripts.
* `WPMW_DOCS` is intentionally **not** set: that variable is for the local
  output-branch worktree workflow, which does not apply on Kaggle. See the
  repository README for details.

In [ ]:
# ── Cell 1: Setup (run once per session) ──────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/billpage/wpmw.git"
REPO_DIR = "/kaggle/working/wpmw"
SRC_DIR  = os.path.join(REPO_DIR, "src")

# Output directory — all scripts read WPMW_OUTPUT via wpmwlib.wpmw_utils.output_path()
os.environ["WPMW_OUTPUT"] = "/kaggle/working"
# WPMW_DOCS is left unset on Kaggle: the output-branch worktree is local-only.
os.environ.pop("WPMW_DOCS", None)

# Clone or pull (single-branch — we don't need the binary 'output' branch)
if os.path.exists(REPO_DIR):
    print("Repo exists — pulling latest …")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)
else:
    print("Cloning repo …")
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", "main", REPO_URL, REPO_DIR],
        check=True,
    )

# Show latest commit
subprocess.run(["git", "log", "--oneline", "-3"], cwd=REPO_DIR)

# Install CuPy (quiet; no-op if already present). Current wpmw scripts do
# not use it, but multi-DOF demos planned for the future will.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Quick GPU check
try:
    import cupy as cp
    print(f"\nGPU ready: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
except Exception as e:
    print(f"\nNo GPU — will fall back to NumPy: {e}")

print(f"\nWPMW_OUTPUT = {os.environ['WPMW_OUTPUT']}")
print(f"WPMW_DOCS   = {os.environ.get('WPMW_DOCS', '(unset)')}")
print(f"Source directory: {SRC_DIR}")
print("Scripts available:", sorted(f for f in os.listdir(SRC_DIR) if f.endswith('.py')))

In [ ]:
# ── Cell 2: Run a script ─────────────────────────────────
#
# Edit SCRIPT and ARGS, then run this cell.
#
# Default: demo_cat_state_microdynamics.py — takes no CLI arguments;
# parameters (N_POSITONS, TIMES, grid sizes) are hard-coded near the top of
# the script. To change them, edit the source and re-run Cell 1 after
# committing/pushing, or edit the file in the working tree directly.
#
SCRIPT  = "demo_cat_state_microdynamics.py"
ARGS    = []
TIMEOUT = 1200  # seconds — 20M-positon run takes several minutes

# ── streaming execution ─────────────────────────────────
import subprocess, sys, os, time

script_path = os.path.join(SRC_DIR, SCRIPT)
assert os.path.exists(script_path), f"Not found: {script_path}"

cmd = [sys.executable, "-u", script_path] + ARGS
print(f">>> {' '.join(cmd)}\n")

t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,           # line-buffered
    cwd=SRC_DIR,         # so `from wpmwlib.wpmw_utils import …` resolves
)

try:
    for line in proc.stdout:
        print(line, end="")
    proc.wait(timeout=TIMEOUT)
except subprocess.TimeoutExpired:
    proc.kill()
    print(f"\n*** TIMEOUT after {TIMEOUT}s ***")

elapsed = time.time() - t0
print(f"\nExit code: {proc.returncode}  ({elapsed:.1f}s)")

In [ ]:
# ── Cell 3: Display PNG outputs + download links ───────────────
from IPython.display import display, Image, FileLink
import glob, os

out = os.environ.get("WPMW_OUTPUT", "/kaggle/working")
pngs = sorted(glob.glob(os.path.join(out, "*.png")), key=os.path.getmtime)
if not pngs:
    print(f"No PNG files found in {out}")
else:
    for p in pngs:
        print(p)
        display(Image(filename=p))
        display(FileLink(p, result_html_prefix="Download: "))

In [ ]:
# ── Cell 4 (optional): Clean up PNGs before next run ────────────
import glob, os
out = os.environ.get("WPMW_OUTPUT", "/kaggle/working")
for p in glob.glob(os.path.join(out, "*.png")):
    os.remove(p)
    print(f"Removed {p}")